# Box medial features - DMF self-supervised finetune

This notebook trains a medial head on frozen Points2Surf features using only the Deep Medial Fields style losses implemented in `source.medial_field`:

- maximality: `ReLU(|phi| - M)^2`
- inscription: `|phi(proj_M(x))| ~= M(x)` with `proj_M(x) = x + grad|phi(x)| * (M(x) - |phi(x)|)`
- orthogonality: `grad M . grad phi ~= 0`

The optional GT medial approximation is for diagnostics only. It is not used by the optimizer or by `compute_medial_losses`. Training queries deliberately mix near-surface and volume samples so the medial field is constrained away from the surface instead of collapsing to tiny radii.




In [ ]:

import importlib
import json
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.utils.data as data
import trimesh
import trimesh.transformations as trafo
from tqdm import tqdm

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'source').is_dir():
    raise RuntimeError('Run this notebook from the points2surf repo root.')
sys.path.insert(0, str(REPO_ROOT))

from source import data_loader, medial_field, points_to_surf_medial, sdf
from source.base import point_cloud
from source.base.utils import torch_load

importlib.reload(medial_field)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Device:', device)



In [ ]:

SHAPE_NAME = 'box'
SHAPE_OBJ = REPO_ROOT / 'data' / 'box.obj'
DATASET_DIR = REPO_ROOT / 'datasets' / 'box_medial'
OUT_DIR = REPO_ROOT / 'models' / 'box_medial_features'

MODEL_DIR = REPO_ROOT / 'models'
MODEL_NAME = 'p2s_vanilla'
MODEL_FILE = MODEL_DIR / f'{MODEL_NAME}_model_149.pth'
PARAM_FILE = MODEL_DIR / f'{MODEL_NAME}_params.pth'

NUM_QUERY_PTS = 4000
NUM_POINTS = 50000
TRAIN_QUERY_LIMIT = None      # set e.g. 512 for a quick CPU experiment
NEAR_SURFACE_QUERY_RATIO = 0.25
FAR_QUERY_PTS_RATIO = 0.35
MIN_INSIDE_SDF_FOR_MEDIAL_TRAIN = 0.03  # avoid near-surface M=0 collapse in medial training
BATCH_SIZE = 8
NEPOCH = 100
LR = 1e-3
RUN_GT_SDF_OVERFIT_DEBUG = True
USE_QUERY_COORDS_FOR_MEDIAL = True
GT_SDF_OVERFIT_STEPS = 120
GT_SDF_OVERFIT_LR = 1e-3
GT_SDF_OVERFIT_WARMSTART_MODEL = True

SLICE_GRID_RES = 96
SLICE_EPSILON = 16
USE_GT_SDF = True  # if True, every medial-training SDF use comes from the GT mesh SDF
INFER_BATCH = 64
EVAL_QUERY_LIMIT = 2000       # set e.g. 2000 if grid evaluation is too slow
MAX_MEDIAL_POINTS = 30000
GT_MEDIAL_AXIS_SURFACE_SAMPLES = 3000
GT_MEDIAL_AXIS_SLICE_WIDTH_CELLS = 2.5
Q_MDF_LEVEL_EPSILON = 0.01
Q_MDF_UNDERPRED_EPSILON = Q_MDF_LEVEL_EPSILON
Q_MDF_METRIC_QUERY_PTS = 1500
Q_MDF_METRIC_EVERY = 1
GT_AXIS_FIELD_METRIC_POINTS = 512
USE_BEST_GT_AXIS_FIELD_HEAD_FOR_VIS = True

DIAGNOSTIC_EVERY = 0        # old opposing-normal diagnostic; new GT-axis metrics run via Q_MDF_METRIC_EVERY
DIAGNOSTIC_QUERY_PTS = 512
GT_DIAGNOSTIC_SAMPLE_COUNT = 7000

LOSS_WEIGHTS = medial_field.get_default_weights()
LOSS_WEIGHTS['maximality'] = 100.0
LOSS_WEIGHTS['inscription'] = 500.0
LOSS_WEIGHTS['orthogonality'] = 0.1
LOSS_WEIGHTS['eikonal'] = 1.0
LOSS_WEIGHTS['surface_reg'] = 1.0
if USE_GT_SDF:
    LOSS_WEIGHTS['eikonal'] = 0.0
    LOSS_WEIGHTS['surface_reg'] = 0.0

OUT_DIR.mkdir(parents=True, exist_ok=True)




In [ ]:

def to_unit_cube(mesh):
    mesh = mesh.copy()
    center = (mesh.bounds[0] + mesh.bounds[1]) * 0.5
    mesh.apply_transform(trafo.translation_matrix(-center))
    mesh.apply_transform(trafo.scale_matrix(1.0 / mesh.extents.max()))
    return mesh

mesh_gt = to_unit_cube(trimesh.load(SHAPE_OBJ, force='mesh'))
if SHAPE_NAME == 'box':
    pts, _ = trimesh.sample.sample_surface(mesh_gt, NUM_POINTS)
    pts = pts.astype(np.float32)
    print(f'Sampled {pts.shape[0]:,} surface points from box mesh.')
else:
    pts = mesh_gt.vertices[:, :3].astype(np.float32)
    if pts.shape[0] > NUM_POINTS:
        pts = pts[np.random.default_rng(42).choice(pts.shape[0], NUM_POINTS, replace=False)]

pts_dir = DATASET_DIR / '04_pts'
qpts_dir = DATASET_DIR / '05_query_pts'
qdist_dir = DATASET_DIR / '05_query_dist'
for d in (pts_dir, qpts_dir, qdist_dir):
    d.mkdir(parents=True, exist_ok=True)
np.save(pts_dir / f'{SHAPE_NAME}.xyz.npy', pts)
(DATASET_DIR / 'testset.txt').write_text(f'{SHAPE_NAME}')

query_pts = medial_field.make_medial_training_queries(
    mesh_gt, NUM_QUERY_PTS,
    near_surface_ratio=NEAR_SURFACE_QUERY_RATIO,
    far_query_pts_ratio=FAR_QUERY_PTS_RATIO,
    seed=42)
if USE_GT_SDF:
    query_phi_gt = -sdf.get_signed_distance(mesh_gt, query_pts)
    inside_query_mask = query_phi_gt < -MIN_INSIDE_SDF_FOR_MEDIAL_TRAIN
    if not np.any(inside_query_mask):
        inside_query_mask = query_phi_gt < -1e-5
        if not np.any(inside_query_mask):
            raise RuntimeError('USE_GT_SDF is on, but no inside training queries were sampled.')
        print(
            'MIN_INSIDE_SDF_FOR_MEDIAL_TRAIN removed every query; '
            'falling back to any inside query.')
    print(
        f'USE_GT_SDF: keeping {inside_query_mask.sum():,}/{len(query_pts):,} inside training queries '
        f'with phi < {-MIN_INSIDE_SDF_FOR_MEDIAL_TRAIN:.4f}')
    query_pts = query_pts[inside_query_mask]
query_sdf_placeholder = np.zeros(len(query_pts), dtype=np.float32)

np.save(qpts_dir / f'{SHAPE_NAME}.ply.npy', query_pts.astype(np.float32))
np.save(qdist_dir / f'{SHAPE_NAME}.ply.npy', query_sdf_placeholder)
print(f'Prepared {len(query_pts)} mixed query points. Placeholder distances are not used by DMF losses.')






In [ ]:

backbone, train_opt = points_to_surf_medial.load_pretrained_backbone(
    str(MODEL_FILE), str(PARAM_FILE), device)
print('Backbone outputs:', train_opt.outputs)

if USE_GT_SDF:
    base_ds = None
    train_ds = None
    loader = None
    print('USE_GT_SDF: skipping PointcloudPatchDataset; using direct GT batches.')
else:
    base_ds = data_loader.PointcloudPatchDataset(
        root=str(DATASET_DIR),
        shape_list_filename='testset.txt',
        points_per_patch=train_opt.points_per_patch,
        patch_radius=train_opt.patch_radius,
        patch_features=train_opt.outputs,
        seed=42,
        center=train_opt.patch_center,
        cache_capacity=1,
        pre_processed_patches=True,
        sub_sample_size=train_opt.sub_sample_size,
        reconstruction=False,
        epsilon=-1,
        uniform_subsample=getattr(train_opt, 'uniform_subsample', 0),
        fixed_subsample=getattr(train_opt, 'fixed_subsample', 0),
        num_workers=0,
    )
    train_ds = base_ds
    if TRAIN_QUERY_LIMIT is not None:
        train_ds = data.Subset(base_ds, np.arange(min(TRAIN_QUERY_LIMIT, len(base_ds))))
    loader = data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

model = points_to_surf_medial.PointsToSurfMedialModel(
    backbone, use_query_coords=USE_QUERY_COORDS_FOR_MEDIAL).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f'Trainable medial-head params: {trainable:,} | frozen backbone params: {frozen:,}')




# Direct GT-SDF batches avoid stale preprocessed patch indices and work for tiny meshes like the box.
direct_gt_rng = np.random.RandomState(4242)
direct_gt_patch_rng = np.random.RandomState(0)
direct_gt_global_rng = np.random.RandomState(1)
direct_gt_kdtree = None
if USE_GT_SDF:
    import scipy.spatial as spatial
    direct_gt_kdtree = spatial.cKDTree(pts)

def make_direct_gt_batch(batch_size=BATCH_SIZE):
    replace = len(query_pts) < batch_size
    ids = direct_gt_rng.choice(len(query_pts), batch_size, replace=replace)
    return medial_field._make_query_batch(
        query_pts[ids].astype(np.float32), pts, direct_gt_kdtree, train_opt,
        direct_gt_patch_rng, direct_gt_global_rng, device)




In [ ]:
# GT-SDF medial-loss gradient smoke test
if USE_GT_SDF:
    smoke_batch = make_direct_gt_batch()
    for k in smoke_batch:
        smoke_batch[k] = smoke_batch[k].to(device)
    model.train()
    smoke_loss, smoke_parts = medial_field.compute_medial_losses_gt_sdf(
        model, smoke_batch, train_opt, mesh_gt, weights=LOSS_WEIGHTS)
    model.zero_grad(set_to_none=True)
    smoke_loss.backward()
    smoke_grad_norm = 0.0
    for p in model.medial_head.parameters():
        if p.grad is not None:
            smoke_grad_norm += float(p.grad.detach().norm().cpu())
    model.zero_grad(set_to_none=True)
    print('GT-SDF medial loss smoke:', {k: float(v.detach().cpu()) for k, v in smoke_parts.items()})
    print(f'GT-SDF medial-head grad norm: {smoke_grad_norm:.6g}')
else:
    print('USE_GT_SDF is off; skipping GT-SDF gradient smoke test.')





In [ ]:
# GT-SDF tiny-batch overfit debug / warm start
if USE_GT_SDF and RUN_GT_SDF_OVERFIT_DEBUG:
    debug_batch = make_direct_gt_batch()
    for k in debug_batch:
        debug_batch[k] = debug_batch[k].to(device)
    debug_model = model if GT_SDF_OVERFIT_WARMSTART_MODEL else points_to_surf_medial.PointsToSurfMedialModel(
        backbone, use_query_coords=USE_QUERY_COORDS_FOR_MEDIAL).to(device)
    debug_optimizer = torch.optim.Adam(
        (p for p in debug_model.parameters() if p.requires_grad), lr=GT_SDF_OVERFIT_LR)
    debug_history = []
    debug_weights = dict(LOSS_WEIGHTS)
    debug_weights['eikonal'] = 0.0
    debug_weights['surface_reg'] = 0.0
    debug_model.train()
    for step in range(GT_SDF_OVERFIT_STEPS):
        debug_optimizer.zero_grad()
        debug_loss, debug_parts = medial_field.compute_medial_losses_gt_sdf(
            debug_model, debug_batch, train_opt, mesh_gt, weights=debug_weights)
        debug_loss.backward()
        debug_optimizer.step()
        debug_history.append(float(debug_loss.detach().cpu()))
    print(f'GT-SDF overfit loss: {debug_history[0]:.6f} -> {debug_history[-1]:.6f}')
    print('Warm-started main model:', GT_SDF_OVERFIT_WARMSTART_MODEL)
    plt.figure(figsize=(6, 3))
    plt.plot(debug_history)
    plt.yscale('log')
    plt.title('GT-SDF tiny-batch overfit debug')
    plt.xlabel('step')
    plt.ylabel('loss')
    plt.tight_layout()
    plt.show()
else:
    print('Skipping GT-SDF tiny-batch overfit debug.')




In [ ]:
medial_field = importlib.reload(medial_field)


# Optional GT-only diagnostics. These points are never passed to compute_medial_losses.
gt_medial_pts = np.empty((0, 3), dtype=np.float32)
diagnostic_queries = np.empty((0, 3), dtype=np.float32)
diagnostic_history = {}
best_diagnostic = None
best_diagnostic_epoch = None

if DIAGNOSTIC_EVERY:
    diagnostic_queries = medial_field.make_medial_training_queries(
        mesh_gt, DIAGNOSTIC_QUERY_PTS,
        near_surface_ratio=NEAR_SURFACE_QUERY_RATIO,
        far_query_pts_ratio=FAR_QUERY_PTS_RATIO,
        seed=123)
    gt_medial_pts, gt_medial_radii = medial_field.approximate_gt_medial_surface_from_mesh(
        mesh_gt, sample_count=GT_DIAGNOSTIC_SAMPLE_COUNT, k=32,
        max_points=MAX_MEDIAL_POINTS, seed=123)
    point_cloud.write_ply(str(OUT_DIR / 'gt_medial_opposing_normals_diagnostic.ply'), gt_medial_pts)
    print(f'Diagnostic queries: {diagnostic_queries.shape[0]:,}')
    print(f'Approx GT diagnostic medial points: {gt_medial_pts.shape[0]:,}')

    initial_diag = medial_field.evaluate_predicted_medial_surface(
        model, train_opt, diagnostic_queries, pts, gt_medial_pts, device,
        out_dir=str(OUT_DIR), tag='initial', batch_size=INFER_BATCH,
        score_percentile=50.0, max_points=MAX_MEDIAL_POINTS,
        mesh_gt_sdf=mesh_gt if USE_GT_SDF else None)
    diagnostic_history['initial'] = initial_diag
    best_diagnostic = initial_diag
    best_diagnostic_epoch = 'initial'
    if initial_diag.get('ply_file'):
        shutil.copyfile(initial_diag['ply_file'], OUT_DIR / 'pred_medial_best_by_diagnostic.ply')
    print('initial diagnostic:', initial_diag)





In [ ]:
# GT medial-axis target plus Q-MDF/projected-center metric setup
if SHAPE_NAME == 'box':
    print('Computing sampled analytic GT medial-axis approximation for box metric...')
    gt_medial_axis_pts, gt_medial_axis_radii = medial_field.approximate_box_medial_axis_from_bounds(
        mesh_gt.bounds, grid_resolution=max(32, SLICE_GRID_RES),
        max_points=MAX_MEDIAL_POINTS, seed=123)
    gt_axis_file = OUT_DIR / 'gt_medial_axis_box.ply'
    gt_axis_source = 'box bounds tie-set'
else:
    print('Computing Voronoi GT medial-axis approximation for metric...')
    gt_medial_axis_pts, gt_medial_axis_radii = medial_field.approximate_medial_axis_voronoi_from_mesh(
        mesh_gt, surface_sample_count=GT_MEDIAL_AXIS_SURFACE_SAMPLES,
        max_points=MAX_MEDIAL_POINTS, seed=123)
    gt_axis_file = OUT_DIR / 'gt_medial_axis_voronoi.ply'
    gt_axis_source = 'surface-sample Voronoi'
point_cloud.write_ply(str(gt_axis_file), gt_medial_axis_pts)
print(f'GT medial-axis points ({gt_axis_source}): {gt_medial_axis_pts.shape[0]:,}')

q_mdf_metric_queries = medial_field.make_medial_training_queries(
    mesh_gt, Q_MDF_METRIC_QUERY_PTS,
    near_surface_ratio=NEAR_SURFACE_QUERY_RATIO,
    far_query_pts_ratio=FAR_QUERY_PTS_RATIO,
    seed=321)
q_mdf_metric_phi_gt = -sdf.get_signed_distance(mesh_gt, q_mdf_metric_queries)
q_mdf_metric_inside = q_mdf_metric_phi_gt < 0.0
q_mdf_metric_queries = q_mdf_metric_queries[q_mdf_metric_inside]
q_mdf_metric_phi_gt = q_mdf_metric_phi_gt[q_mdf_metric_inside]
print(f'Metric inside query points: {q_mdf_metric_queries.shape[0]:,}')

q_mdf_metric_history = []
projected_axis_metric_history = []
gt_axis_field_metric_history = []
best_gt_axis_field = None
best_gt_axis_field_epoch = None

def metric_epoch_from_tag(tag):
    return -1 if tag == 'initial' else int(tag.split('_')[-1])

def evaluate_q_mdf_metric(tag):
    if gt_medial_axis_pts.shape[0] == 0 or q_mdf_metric_queries.shape[0] == 0:
        result = {
            'tag': tag,
            'epoch': metric_epoch_from_tag(tag),
            'chamfer_l2': np.inf,
            'pred_to_gt': np.inf,
            'gt_to_pred': np.inf,
            'q_mdf_level_set_count': 0,
            'q_mdf_underpred_fraction': None,
            'q_mdf_valid_band_fraction': None,
        }
        q_pts = np.empty((0, 3), dtype=np.float32)
    else:
        if USE_GT_SDF:
            med_metric = medial_field.predict_medial_on_queries(
                model, train_opt, q_mdf_metric_queries, pts, device, batch_size=INFER_BATCH)
            phi_metric = q_mdf_metric_phi_gt
        else:
            phi_metric, med_metric = medial_field.predict_fields_on_queries(
                model, train_opt, q_mdf_metric_queries, pts, device, batch_size=INFER_BATCH)
        metrics, q_pts = medial_field.q_mdf_level_set_metrics(
            q_mdf_metric_queries, phi_metric, med_metric, gt_medial_axis_pts,
            epsilon=Q_MDF_LEVEL_EPSILON, inside_only=True,
            max_points=MAX_MEDIAL_POINTS, rng=np.random.RandomState(123),
            require_valid=True)
        result = dict(metrics)
        result['tag'] = tag
        result['epoch'] = metric_epoch_from_tag(tag)
    point_cloud.write_ply(str(OUT_DIR / f'q_mdf_level_set_{tag}.ply'), q_pts)
    q_mdf_metric_history.append(result)
    under = result.get('q_mdf_underpred_fraction')
    band = result.get('q_mdf_valid_band_fraction')
    under_txt = 'nan' if under is None else f'{under:.3f}'
    band_txt = 'nan' if band is None else f'{band:.3f}'
    print(
        f"{tag} Q-MDF band metric: chamfer={result['chamfer_l2']:.6g}, "
        f"count={result['q_mdf_level_set_count']}, band_frac={band_txt}, underpred_frac={under_txt}")
    return result

def evaluate_projected_axis_metric(tag):
    if gt_medial_axis_pts.shape[0] == 0 or q_mdf_metric_queries.shape[0] == 0:
        result = {
            'tag': tag,
            'epoch': metric_epoch_from_tag(tag),
            'chamfer_l2': np.inf,
            'pred_to_gt': np.inf,
            'gt_to_pred': np.inf,
            'count': 0,
            'inside_eval_count': 0,
            'mean_abs_score': None,
        }
    else:
        result = medial_field.evaluate_predicted_medial_surface(
            model, train_opt, q_mdf_metric_queries, pts, gt_medial_axis_pts, device,
            out_dir=str(OUT_DIR), tag=f'projected_axis_{tag}', batch_size=INFER_BATCH,
            score_percentile=None, max_points=MAX_MEDIAL_POINTS,
            mesh_gt_sdf=mesh_gt if USE_GT_SDF else None)
        result['tag'] = tag
        result['epoch'] = metric_epoch_from_tag(tag)
    projected_axis_metric_history.append(result)
    mean_score = result.get('mean_abs_score')
    mean_score_txt = 'nan' if mean_score is None else f'{mean_score:.6g}'
    print(
        f"{tag} projected-center metric: chamfer={result['chamfer_l2']:.6g}, "
        f"count={result['count']}, mean_abs(M-|phi|)={mean_score_txt}")
    return result

def evaluate_gt_axis_field_metric(tag, save_best=False):
    global best_gt_axis_field, best_gt_axis_field_epoch
    result = medial_field.evaluate_medial_field_on_gt_axis(
        model, train_opt, gt_medial_axis_pts, gt_medial_axis_radii, pts, device,
        batch_size=INFER_BATCH, max_points=GT_AXIS_FIELD_METRIC_POINTS,
        rng=np.random.RandomState(456), epsilon=Q_MDF_LEVEL_EPSILON)
    result['tag'] = tag
    result['epoch'] = metric_epoch_from_tag(tag)
    gt_axis_field_metric_history.append(result)
    mae = result['gt_axis_residual_mae']
    if np.isfinite(mae) and (best_gt_axis_field is None or mae < best_gt_axis_field['gt_axis_residual_mae']):
        best_gt_axis_field = dict(result)
        best_gt_axis_field_epoch = tag
        if save_best:
            torch.save(model.medial_head.state_dict(), OUT_DIR / 'best_medial_head_by_gt_axis_field.pth')
            print('Saved best GT-axis field checkpoint:', best_gt_axis_field_epoch)
    print(
        f"{tag} GT-axis field residual: mae={result['gt_axis_residual_mae']:.6g}, "
        f"rmse={result['gt_axis_residual_rmse']:.6g}, "
        f"valid_frac={result['gt_axis_valid_fraction']:.3f}, "
        f"underpred_frac={result['gt_axis_underpred_fraction']:.3f}")
    return result

evaluate_q_mdf_metric('initial')
evaluate_projected_axis_metric('initial')
evaluate_gt_axis_field_metric('initial', save_best=False)


In [ ]:
optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=LR)
history = {k: [] for k in ['Total', 'Maximality', 'Inscription', 'Orthogonality', 'Eikonal', 'Surface Regularizer']}

gt_train_rng = np.random.RandomState(4242)
gt_patch_rng = np.random.RandomState(0)
gt_global_rng = np.random.RandomState(1)
gt_kdtree = None
if USE_GT_SDF:
    import scipy.spatial as spatial
    gt_kdtree = spatial.cKDTree(pts)
    gt_steps_per_epoch = max(1, int(np.ceil(len(query_pts) / BATCH_SIZE)))
    print(f'USE_GT_SDF direct training: {len(query_pts):,} inside queries, {gt_steps_per_epoch} steps/epoch')
else:
    gt_steps_per_epoch = None

def make_gt_training_batch():
    replace = len(query_pts) < BATCH_SIZE
    ids = gt_train_rng.choice(len(query_pts), BATCH_SIZE, replace=replace)
    return medial_field._make_query_batch(
        query_pts[ids].astype(np.float32), pts, gt_kdtree, train_opt,
        gt_patch_rng, gt_global_rng, device)

train_start_time = time.time()
model.train()
for epoch in range(NEPOCH):
    epoch_parts = {k: [] for k in history}
    t = epoch / max(NEPOCH - 1, 1)
    if USE_GT_SDF:
        batch_iter = (make_gt_training_batch() for _ in range(gt_steps_per_epoch))
        progress = tqdm(batch_iter, total=gt_steps_per_epoch, desc=f'epoch {epoch}', leave=False)
    else:
        progress = tqdm(loader, desc=f'epoch {epoch}', leave=False)

    for batch in progress:
        for k in batch:
            batch[k] = batch[k].to(device)
        if USE_GT_SDF:
            loss, parts = medial_field.compute_medial_losses_gt_sdf(
                model, batch, train_opt, mesh_gt, weights=LOSS_WEIGHTS, t=t)
        else:
            loss, parts = medial_field.compute_medial_losses(
                model, batch, train_opt, weights=LOSS_WEIGHTS, t=t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        for k in history:
            if k in parts:
                epoch_parts[k].append(float(parts[k].detach().cpu()))
    for k in history:
        history[k].append(float(np.mean(epoch_parts[k])) if epoch_parts[k] else 0.0)
    print(
        f"epoch {epoch:03d} total={history['Total'][-1]:.4f} "
        f"max={history['Maximality'][-1]:.4f} ins={history['Inscription'][-1]:.4f} "
        f"orth={history['Orthogonality'][-1]:.4f}")

    tag = f'epoch_{epoch:03d}'
    if DIAGNOSTIC_EVERY and ((epoch + 1) % DIAGNOSTIC_EVERY == 0):
        model.eval()
        diag = medial_field.evaluate_predicted_medial_surface(
            model, train_opt, diagnostic_queries, pts, gt_medial_pts, device,
            out_dir=str(OUT_DIR), tag=tag, batch_size=INFER_BATCH,
            score_percentile=50.0, max_points=MAX_MEDIAL_POINTS,
            mesh_gt_sdf=mesh_gt if USE_GT_SDF else None)
        diagnostic_history[tag] = diag
        print(f'{tag} diagnostic:', diag)
        if best_diagnostic is None or diag['chamfer_l2'] < best_diagnostic['chamfer_l2']:
            best_diagnostic = diag
            best_diagnostic_epoch = tag
            torch.save(model.medial_head.state_dict(), OUT_DIR / 'best_medial_head_by_diagnostic.pth')
            if diag.get('ply_file'):
                shutil.copyfile(diag['ply_file'], OUT_DIR / 'pred_medial_best_by_diagnostic.ply')
            print('Saved best diagnostic checkpoint:', best_diagnostic_epoch)
        if Q_MDF_METRIC_EVERY and ((epoch + 1) % Q_MDF_METRIC_EVERY == 0):
            evaluate_q_mdf_metric(tag)
            evaluate_projected_axis_metric(tag)
            evaluate_gt_axis_field_metric(tag, save_best=True)
        model.train()
    elif Q_MDF_METRIC_EVERY and ((epoch + 1) % Q_MDF_METRIC_EVERY == 0):
        model.eval()
        evaluate_q_mdf_metric(tag)
        evaluate_projected_axis_metric(tag)
        evaluate_gt_axis_field_metric(tag, save_best=True)
        model.train()

train_seconds = time.time() - train_start_time
torch.save(model.medial_head.state_dict(), OUT_DIR / 'medial_head.pth')
print('Saved', OUT_DIR / 'medial_head.pth')
if best_diagnostic is not None:
    print('Best GT-only diagnostic checkpoint:', best_diagnostic_epoch, best_diagnostic)
if best_gt_axis_field is not None:
    print('Best GT-axis field checkpoint:', best_gt_axis_field_epoch, best_gt_axis_field)

training_summary = {
    'shape_name': SHAPE_NAME,
    'use_gt_sdf': USE_GT_SDF,
    'use_query_coords_for_medial': USE_QUERY_COORDS_FOR_MEDIAL,
    'lr': LR,
    'nepoch': NEPOCH,
    'loss_weights': LOSS_WEIGHTS,
    'train_seconds': train_seconds,
    'history': history,
    'q_mdf_metric_history': q_mdf_metric_history,
    'projected_axis_metric_history': projected_axis_metric_history,
    'gt_axis_field_metric_history': gt_axis_field_metric_history,
    'best_gt_axis_field_epoch': best_gt_axis_field_epoch,
    'best_gt_axis_field': best_gt_axis_field,
    'best_diagnostic_epoch': best_diagnostic_epoch,
    'best_diagnostic': best_diagnostic,
}
summary_file = OUT_DIR / 'medial_training_summary.json'
summary_file.write_text(json.dumps(training_summary, indent=2))
print('Saved training summary:', summary_file)

best_gt_axis_head_file = OUT_DIR / 'best_medial_head_by_gt_axis_field.pth'
if USE_BEST_GT_AXIS_FIELD_HEAD_FOR_VIS and best_gt_axis_head_file.is_file():
    model.medial_head.load_state_dict(torch_load(best_gt_axis_head_file, map_location=device))
    model.eval()
    print('Loaded best GT-axis field head for downstream eval/visualization:', best_gt_axis_head_file)
elif USE_BEST_GT_AXIS_FIELD_HEAD_FOR_VIS:
    print('Best GT-axis field head not found; downstream eval/visualization will use final head.')




In [ ]:
loss_keys = ['Total', 'Maximality', 'Inscription', 'Orthogonality']
loss_weight_lookup = {
    'Maximality': LOSS_WEIGHTS.get('maximality', 0.0),
    'Inscription': LOSS_WEIGHTS.get('inscription', 0.0),
    'Orthogonality': LOSS_WEIGHTS.get('orthogonality', 0.0),
    'Eikonal': LOSS_WEIGHTS.get('eikonal', 0.0),
    'Surface Regularizer': LOSS_WEIGHTS.get('surface_reg', 0.0),
}
weighted_history = {
    k: (np.asarray(history[k], dtype=np.float64) * loss_weight_lookup[k]).tolist()
    for k in ['Maximality', 'Inscription', 'Orthogonality']
    if k in history
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
ax = axes[0]
for k in loss_keys:
    vals = np.asarray(history[k], dtype=np.float64)
    vals = np.where(vals > 0.0, vals, np.nan)
    ax.plot(vals, marker='o', label=k)
ax.set_yscale('log')
ax.set_xlabel('epoch')
ax.set_ylabel('raw loss value')
ax.legend()
ax.set_title('Raw DMF loss terms')
ax.grid(True, which='both', alpha=0.25)

ax = axes[1]
total_vals = np.asarray(history['Total'], dtype=np.float64)
total_vals = np.where(total_vals > 0.0, total_vals, np.nan)
ax.plot(total_vals, marker='o', color='black', linewidth=2.0, label='Total')
for k, vals in weighted_history.items():
    vals = np.asarray(vals, dtype=np.float64)
    vals = np.where(vals > 0.0, vals, np.nan)
    ax.plot(vals, marker='o', label=f'{k} x {loss_weight_lookup[k]:g}')
ax.set_yscale('log')
ax.set_xlabel('epoch')
ax.set_ylabel('weighted contribution')
ax.legend()
ax.set_title('What actually drives Total')
ax.grid(True, which='both', alpha=0.25)
plt.show()

print('Raw loss first -> last summary:')
for k in loss_keys:
    vals = np.asarray(history[k], dtype=np.float64)
    finite = vals[np.isfinite(vals)]
    if finite.size == 0:
        print(f'  {k:14s}: no finite values')
        continue
    first, last = finite[0], finite[-1]
    ratio = last / first if first != 0.0 else np.nan
    direction = 'down' if last < first else 'up/flat'
    print(f'  {k:14s}: {first:.6g} -> {last:.6g}  ratio={ratio:.3g}  ({direction})')

print('Weighted contribution first -> last summary:')
weighted_summary = {'Total': history['Total'], **weighted_history}
for k, vals in weighted_summary.items():
    vals = np.asarray(vals, dtype=np.float64)
    finite = vals[np.isfinite(vals)]
    if finite.size == 0:
        print(f'  {k:14s}: no finite values')
        continue
    first, last = finite[0], finite[-1]
    ratio = last / first if first != 0.0 else np.nan
    direction = 'down' if last < first else 'up/flat'
    print(f'  {k:14s}: {first:.6g} -> {last:.6g}  ratio={ratio:.3g}  ({direction})')


In [ ]:
# Medial-axis eval metrics vs epochs
has_q_mdf = bool(q_mdf_metric_history)
has_projected = bool(projected_axis_metric_history)
has_gt_axis_field = bool(gt_axis_field_metric_history)
if has_q_mdf or has_projected or has_gt_axis_field:
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.4), constrained_layout=True)

    if has_q_mdf:
        metric_epochs = [row['epoch'] for row in q_mdf_metric_history]
        metric_labels = ['init' if e < 0 else str(e) for e in metric_epochs]
        chamfer_values = [row['chamfer_l2'] for row in q_mdf_metric_history]
        pred_to_gt_values = [row['pred_to_gt'] for row in q_mdf_metric_history]
        gt_to_pred_values = [row['gt_to_pred'] for row in q_mdf_metric_history]
        underpred_values = [row.get('q_mdf_underpred_fraction', np.nan) for row in q_mdf_metric_history]
        band_values = [row.get('q_mdf_valid_band_fraction', np.nan) for row in q_mdf_metric_history]
        ax = axes[0]
        ax.plot(metric_epochs, chamfer_values, marker='o', label='Chamfer L2')
        ax.plot(metric_epochs, pred_to_gt_values, marker='o', label='Q-MDF band -> GT')
        ax.plot(metric_epochs, gt_to_pred_values, marker='o', label='GT -> Q-MDF band')
        ax.set_xticks(metric_epochs)
        ax.set_xticklabels(metric_labels)
        ax.set_xlabel('epoch')
        ax.set_ylabel('3D distance metric')
        ax.set_title('Q-MDF valid narrow band')
        ax.grid(True, which='both', alpha=0.25)
        ax.legend(loc='upper left', fontsize=8)
        ax2 = ax.twinx()
        ax2.plot(metric_epochs, underpred_values, marker='x', linestyle='--', color='tab:red', label='underpred fraction')
        ax2.plot(metric_epochs, band_values, marker='x', linestyle='--', color='tab:green', label='valid band fraction')
        ax2.set_ylabel('sample fraction')
        ax2.set_ylim(0.0, 1.0)
        ax2.legend(loc='upper right', fontsize=8)
    else:
        axes[0].axis('off')
        axes[0].text(0.5, 0.5, 'No Q-MDF metric history', ha='center', va='center')

    if has_projected:
        projected_epochs = [row['epoch'] for row in projected_axis_metric_history]
        projected_labels = ['init' if e < 0 else str(e) for e in projected_epochs]
        projected_chamfer = [row['chamfer_l2'] for row in projected_axis_metric_history]
        projected_pred_to_gt = [row['pred_to_gt'] for row in projected_axis_metric_history]
        projected_gt_to_pred = [row['gt_to_pred'] for row in projected_axis_metric_history]
        projected_score = [row.get('mean_abs_score', np.nan) for row in projected_axis_metric_history]
        ax = axes[1]
        ax.plot(projected_epochs, projected_chamfer, marker='o', label='Chamfer L2')
        ax.plot(projected_epochs, projected_pred_to_gt, marker='o', label='projected centers -> GT')
        ax.plot(projected_epochs, projected_gt_to_pred, marker='o', label='GT -> projected centers')
        ax.set_xticks(projected_epochs)
        ax.set_xticklabels(projected_labels)
        ax.set_xlabel('epoch')
        ax.set_ylabel('3D distance metric')
        ax.set_title('Projected medial centers')
        ax.grid(True, which='both', alpha=0.25)
        ax.legend(loc='upper left', fontsize=8)
        ax2 = ax.twinx()
        ax2.plot(projected_epochs, projected_score, marker='x', linestyle='--', color='tab:purple', label='mean |M-|phi||')
        ax2.set_ylabel('field residual')
        ax2.legend(loc='upper right', fontsize=8)
    else:
        axes[1].axis('off')
        axes[1].text(0.5, 0.5, 'No projected-center metric history', ha='center', va='center')

    if has_gt_axis_field:
        axis_epochs = [row['epoch'] for row in gt_axis_field_metric_history]
        axis_labels = ['init' if e < 0 else str(e) for e in axis_epochs]
        axis_mae = [row['gt_axis_residual_mae'] for row in gt_axis_field_metric_history]
        axis_rmse = [row['gt_axis_residual_rmse'] for row in gt_axis_field_metric_history]
        axis_bias = [row['gt_axis_residual_bias'] for row in gt_axis_field_metric_history]
        axis_valid = [row['gt_axis_valid_fraction'] for row in gt_axis_field_metric_history]
        axis_under = [row['gt_axis_underpred_fraction'] for row in gt_axis_field_metric_history]
        ax = axes[2]
        ax.plot(axis_epochs, axis_mae, marker='o', label='MAE M-r')
        if best_gt_axis_field is not None:
            best_epoch = best_gt_axis_field['epoch']
            best_mae = best_gt_axis_field['gt_axis_residual_mae']
            ax.scatter([best_epoch], [best_mae], s=90, marker='*', color='gold', edgecolor='black', zorder=5, label='best MAE')
        ax.plot(axis_epochs, axis_rmse, marker='o', label='RMSE M-r')
        ax.plot(axis_epochs, np.abs(axis_bias), marker='o', label='|bias|')
        ax.set_xticks(axis_epochs)
        ax.set_xticklabels(axis_labels)
        ax.set_xlabel('epoch')
        ax.set_ylabel('radius residual')
        ax.set_title('Field evaluated on GT axis')
        ax.grid(True, which='both', alpha=0.25)
        ax.legend(loc='upper left', fontsize=8)
        ax2 = ax.twinx()
        ax2.plot(axis_epochs, axis_valid, marker='x', linestyle='--', color='tab:green', label='valid fraction')
        ax2.plot(axis_epochs, axis_under, marker='x', linestyle='--', color='tab:red', label='underpred fraction')
        ax2.set_ylabel('sample fraction')
        ax2.set_ylim(0.0, 1.0)
        ax2.legend(loc='upper right', fontsize=8)
    else:
        axes[2].axis('off')
        axes[2].text(0.5, 0.5, 'No GT-axis field metric history', ha='center', va='center')

    metric_plot = OUT_DIR / 'medial_axis_metrics_vs_epochs.png'
    fig.savefig(metric_plot, dpi=180)
    plt.show()
    print('Saved', metric_plot)
else:
    print('No metric history yet. Run training/evaluation cells first.')


In [ ]:

model.eval()
grid_pts = sdf.get_voxel_centers_grid_smaller_pc(
    pts, grid_resolution=SLICE_GRID_RES, distance_threshold_vs=SLICE_EPSILON)
if EVAL_QUERY_LIMIT is not None and grid_pts.shape[0] > EVAL_QUERY_LIMIT:
    eval_rng = np.random.default_rng(123)
    grid_pts = grid_pts[eval_rng.choice(grid_pts.shape[0], EVAL_QUERY_LIMIT, replace=False)]
print(f'Grid query points near bunny: {grid_pts.shape[0]:,}')

if USE_GT_SDF:
    phi_grid = -sdf.get_signed_distance(mesh_gt, grid_pts)
    med_grid = medial_field.predict_medial_on_queries(
        model, train_opt, grid_pts, pts, device, batch_size=INFER_BATCH)
else:
    phi_grid, med_grid = medial_field.predict_fields_on_queries(
        model, train_opt, grid_pts, pts, device, batch_size=INFER_BATCH)
score_grid = medial_field.medial_level_score(phi_grid, med_grid)

near_sheet_pts, near_sheet_score = medial_field.select_predicted_medial_points(
    grid_pts, phi_grid, med_grid, inside_only=True, score_percentile=2.0,
    max_points=MAX_MEDIAL_POINTS)
print(f'Near-zero M-|phi| sheet samples: {near_sheet_pts.shape[0]:,}')

if USE_GT_SDF:
    projected_pts, projected_phi, projected_med = medial_field.project_queries_to_medial_surface_gt_sdf(
        model, train_opt, mesh_gt, grid_pts, pts, device, batch_size=INFER_BATCH,
        inside_only=True, score_percentile=25.0, max_points=MAX_MEDIAL_POINTS)
else:
    projected_pts, projected_phi, projected_med = medial_field.project_queries_to_medial_surface(
        model, train_opt, grid_pts, pts, device, batch_size=INFER_BATCH,
        inside_only=True, score_percentile=25.0, max_points=MAX_MEDIAL_POINTS)
print(f'Projected predicted medial centers: {projected_pts.shape[0]:,}')

point_cloud.write_ply(str(OUT_DIR / 'pred_medial_near_sheet.ply'), near_sheet_pts)
point_cloud.write_ply(str(OUT_DIR / 'pred_medial_projected.ply'), projected_pts)
print('Saved predicted medial point clouds to', OUT_DIR)





In [ ]:
def make_field_slice_queries(pts, axis='z', value=None, grid_res=96, padding=0.08):
    axis_to_dim = {'x': 0, 'y': 1, 'z': 2}
    dim = axis_to_dim[axis]
    axes = [i for i in range(3) if i != dim]

    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    span = maxs - mins
    lo = mins[axes] - padding * span[axes]
    hi = maxs[axes] + padding * span[axes]
    value = float(0.5 * (mins[dim] + maxs[dim]) if value is None else value)

    a = np.linspace(lo[0], hi[0], grid_res, dtype=np.float32)
    b = np.linspace(lo[1], hi[1], grid_res, dtype=np.float32)
    aa, bb = np.meshgrid(a, b, indexing='xy')
    queries = np.zeros((grid_res * grid_res, 3), dtype=np.float32)
    queries[:, axes[0]] = aa.reshape(-1)
    queries[:, axes[1]] = bb.reshape(-1)
    queries[:, dim] = value
    extent = [float(a.min()), float(a.max()), float(b.min()), float(b.max())]
    return queries, axes, dim, value, extent


def add_slice_panel(fig, ax, field, title, axes, dim, value, extent,
                    cmap='viridis', vmin=None, vmax=None, cbar_label='field',
                    draw_zero_contour=False, contour_levels=None, contour_color='black',
                    contour_field=None):
    if hasattr(cmap, 'copy'):
        cmap = cmap.copy()
        cmap.set_bad(color=(1.0, 1.0, 1.0, 0.0))
    im = ax.imshow(np.ma.masked_invalid(field), origin='lower', extent=extent, cmap=cmap, vmin=vmin, vmax=vmax)
    ny, nx = field.shape
    xs = np.linspace(extent[0], extent[1], nx)
    ys = np.linspace(extent[2], extent[3], ny)
    if draw_zero_contour and np.nanmin(field) <= 0.0 <= np.nanmax(field):
        ax.contour(xs, ys, field, levels=[0.0], colors='black', linewidths=1.2)
    if contour_levels is not None:
        contour_source = field if contour_field is None else contour_field
        finite = contour_source[np.isfinite(contour_source)]
        if finite.size:
            valid_levels = [level for level in contour_levels if finite.min() <= level <= finite.max()]
            if valid_levels:
                ax.contour(xs, ys, contour_source, levels=valid_levels, colors=contour_color, linewidths=1.2)
    thickness = max((pts[:, dim].max() - pts[:, dim].min()) / SLICE_GRID_RES * 1.5, 1e-4)
    slice_mask = np.abs(pts[:, dim] - value) <= thickness
    if np.any(slice_mask):
        ax.scatter(pts[slice_mask, axes[0]], pts[slice_mask, axes[1]], s=0.25, c='black', alpha=0.3, linewidths=0)
    ax.set_aspect('equal')
    ax.set_xlabel('xyz'[axes[0]])
    ax.set_ylabel('xyz'[axes[1]])
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(cbar_label)
    return im


def plot_medial_axis_slice_panel(fig, ax, medial_pts, medial_radii, title, axes, dim, value, extent, cell_size):
    ax.set_aspect('equal')
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    ax.set_xlabel('xyz'[axes[0]])
    ax.set_ylabel('xyz'[axes[1]])
    ax.set_title(title)
    thickness = max(GT_MEDIAL_AXIS_SLICE_WIDTH_CELLS * cell_size, 1e-4)
    surface_mask = np.abs(pts[:, dim] - value) <= thickness
    if np.any(surface_mask):
        ax.scatter(pts[surface_mask, axes[0]], pts[surface_mask, axes[1]], s=0.25, c='0.75', alpha=0.35, linewidths=0)
    if medial_pts.shape[0] == 0:
        ax.text(0.5, 0.5, 'no GT medial-axis points', ha='center', va='center', transform=ax.transAxes)
        return None
    slab_mask = np.abs(medial_pts[:, dim] - value) <= thickness
    if not np.any(slab_mask):
        ax.text(0.5, 0.5, 'no GT medial-axis points in slice slab', ha='center', va='center', transform=ax.transAxes)
        return None
    sc = ax.scatter(
        medial_pts[slab_mask, axes[0]], medial_pts[slab_mask, axes[1]],
        c=medial_radii[slab_mask], cmap='plasma', s=7, alpha=0.95, linewidths=0)
    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('GT medial radius')
    return sc


medial_field = importlib.reload(medial_field)
q_mdf_cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    'q_mdf_positive_rd_bu_r', plt.get_cmap('RdBu_r')(np.linspace(0.5, 1.0, 256)))

slice_queries, slice_axes, slice_dim, slice_value, slice_extent = make_field_slice_queries(
    pts, axis='z', value=None, grid_res=SLICE_GRID_RES)
slice_cell_size = max(
    (slice_extent[1] - slice_extent[0]) / max(SLICE_GRID_RES - 1, 1),
    (slice_extent[3] - slice_extent[2]) / max(SLICE_GRID_RES - 1, 1))
print(f'Evaluating {slice_queries.shape[0]:,} points on z={slice_value:.4f} slice')

slice_phi, slice_medial = medial_field.predict_fields_on_queries(
    model, train_opt, slice_queries, pts, device, batch_size=INFER_BATCH)
# trimesh uses positive-inside; flip so GT matches our inside-negative SDF convention.
slice_phi_gt = -sdf.get_signed_distance(mesh_gt, slice_queries)
sdf_slice = slice_phi.reshape(SLICE_GRID_RES, SLICE_GRID_RES)
sdf_gt_slice = slice_phi_gt.reshape(SLICE_GRID_RES, SLICE_GRID_RES)
medial_slice = slice_medial.reshape(SLICE_GRID_RES, SLICE_GRID_RES)
q_mdf_phi = slice_phi_gt if USE_GT_SDF else slice_phi
q_mdf_source = 'GT SDF' if USE_GT_SDF else 'predicted SDF'
q_mdf_residual_slice = medial_field.medial_level_score(q_mdf_phi, slice_medial).reshape(SLICE_GRID_RES, SLICE_GRID_RES)
q_mdf_slice = np.maximum(q_mdf_residual_slice, 0.0)  # Q-MDF is UDF-like only where M >= |SDF|.
inside_slice_mask = sdf_gt_slice < 0.0
q_mdf_underpred_slice = q_mdf_residual_slice < -Q_MDF_UNDERPRED_EPSILON
q_mdf_valid_slice_mask = inside_slice_mask & ~q_mdf_underpred_slice
q_mdf_slice_inside = np.where(q_mdf_valid_slice_mask, q_mdf_slice, np.nan)
q_mdf_residual_slice_inside = np.where(inside_slice_mask, q_mdf_residual_slice, np.nan)
q_mdf_underpred_slice_inside = np.where(inside_slice_mask, q_mdf_underpred_slice.astype(np.float32), np.nan)
underpred_frac_slice = float(np.nanmean(q_mdf_underpred_slice_inside)) if np.any(inside_slice_mask) else np.nan
print(f'Q-MDF computed from {q_mdf_source}; underpredicted inside-slice fraction={underpred_frac_slice:.3f}')

sdf_abs = np.abs(np.concatenate([
    sdf_slice[np.isfinite(sdf_slice)],
    sdf_gt_slice[np.isfinite(sdf_gt_slice)],
]))
sdf_vmax = float(np.percentile(sdf_abs, 95)) if sdf_abs.size else 1.0
sdf_vmax = max(sdf_vmax, 1e-6)

medial_vals = medial_slice[np.isfinite(medial_slice)]
medial_vmax = float(np.percentile(medial_vals, 98)) if medial_vals.size else None

q_vals = q_mdf_slice_inside[np.isfinite(q_mdf_slice_inside)]
q_vmax = float(np.percentile(q_vals, 95)) if q_vals.size else 1.0
q_vmax = max(q_vmax, 1e-6)

if 'gt_medial_axis_pts' not in globals() or 'gt_medial_axis_radii' not in globals():
    if SHAPE_NAME == 'box':
        print('Computing sampled analytic GT medial-axis approximation for box...')
        gt_medial_axis_pts, gt_medial_axis_radii = medial_field.approximate_box_medial_axis_from_bounds(
            mesh_gt.bounds, grid_resolution=max(32, SLICE_GRID_RES),
            max_points=MAX_MEDIAL_POINTS, seed=123)
        point_cloud.write_ply(str(OUT_DIR / 'gt_medial_axis_box.ply'), gt_medial_axis_pts)
    else:
        print('Computing Voronoi GT medial-axis approximation...')
        gt_medial_axis_pts, gt_medial_axis_radii = medial_field.approximate_medial_axis_voronoi_from_mesh(
            mesh_gt, surface_sample_count=GT_MEDIAL_AXIS_SURFACE_SAMPLES,
            max_points=MAX_MEDIAL_POINTS, seed=123)
        point_cloud.write_ply(str(OUT_DIR / 'gt_medial_axis_voronoi.ply'), gt_medial_axis_pts)
print(f'GT medial-axis points: {gt_medial_axis_pts.shape[0]:,}')

fig, axes = plt.subplots(2, 3, figsize=(17, 10.0), constrained_layout=True)
add_slice_panel(
    fig, axes[0, 0], sdf_slice, 'Predicted SDF field', slice_axes, slice_dim, slice_value, slice_extent,
    cmap='RdBu_r', vmin=-sdf_vmax, vmax=sdf_vmax, cbar_label='signed distance',
    draw_zero_contour=True)
add_slice_panel(
    fig, axes[1, 0], sdf_gt_slice, 'GT SDF field', slice_axes, slice_dim, slice_value, slice_extent,
    cmap='RdBu_r', vmin=-sdf_vmax, vmax=sdf_vmax, cbar_label='signed distance',
    draw_zero_contour=True)
add_slice_panel(
    fig, axes[0, 1], medial_slice, 'Medial field', slice_axes, slice_dim, slice_value, slice_extent,
    cmap='RdBu_r', vmin=0.0, vmax=medial_vmax, cbar_label='medial field M',
    contour_levels=[0.0], contour_color='white', contour_field=sdf_slice)
add_slice_panel(
    fig, axes[0, 2], q_mdf_slice_inside, f'Valid Q-MDF({q_mdf_source})', slice_axes, slice_dim, slice_value, slice_extent,
    cmap=q_mdf_cmap, vmin=0.0, vmax=q_vmax, cbar_label='Q-MDF / UDF-like distance',
    contour_levels=[0.0], contour_color='black', contour_field=q_mdf_residual_slice_inside)
add_slice_panel(
    fig, axes[1, 1], q_mdf_underpred_slice_inside, 'Underpredicted region: M < |SDF|',
    slice_axes, slice_dim, slice_value, slice_extent, cmap='Reds', vmin=0.0, vmax=1.0,
    cbar_label='underpredicted mask')
plot_medial_axis_slice_panel(
    fig, axes[1, 2], gt_medial_axis_pts, gt_medial_axis_radii,
    'GT medial axis (Voronoi approximation)', slice_axes, slice_dim, slice_value,
    slice_extent, slice_cell_size)
fig.suptitle(f'Box central z-slice at z={slice_value:.4f}', y=1.02)
combined_slice_plot = OUT_DIR / 'box_sdf_medial_q_mdf_field_slices.png'
fig.savefig(combined_slice_plot, dpi=180, bbox_inches='tight')
plt.show()
print('Saved', combined_slice_plot)

# Automatic timestamped training snapshot
snapshot_timestamp = time.strftime('%Y%m%d_%H%M%S')
snapshot_dir = OUT_DIR / 'snapshots' / f'{SHAPE_NAME}_{snapshot_timestamp}'
snapshot_dir.mkdir(parents=True, exist_ok=False)

# Save the exact model currently used by the downstream visualizations. If
# USE_BEST_GT_AXIS_FIELD_HEAD_FOR_VIS is True, this is the best GT-axis head.
torch.save(model.medial_head.state_dict(), snapshot_dir / 'medial_head_current.pth')
torch.save(model.state_dict(), snapshot_dir / 'points_to_surf_medial_model_state_dict.pth')

snapshot_manifest = {
    'snapshot_timestamp': snapshot_timestamp,
    'shape_name': SHAPE_NAME,
    'shape_obj': str(SHAPE_OBJ),
    'source_out_dir': str(OUT_DIR),
    'use_gt_sdf': USE_GT_SDF,
    'use_query_coords_for_medial': USE_QUERY_COORDS_FOR_MEDIAL,
    'use_best_gt_axis_field_head_for_vis': USE_BEST_GT_AXIS_FIELD_HEAD_FOR_VIS,
    'best_gt_axis_field_epoch': best_gt_axis_field_epoch if 'best_gt_axis_field_epoch' in globals() else None,
    'best_gt_axis_field': best_gt_axis_field if 'best_gt_axis_field' in globals() else None,
    'lr': LR,
    'nepoch': NEPOCH,
    'loss_weights': LOSS_WEIGHTS,
    'model_file': str(MODEL_FILE),
    'param_file': str(PARAM_FILE),
}
(snapshot_dir / 'snapshot_manifest.json').write_text(json.dumps(snapshot_manifest, indent=2, default=str))

snapshot_artifacts = [
    OUT_DIR / 'medial_head.pth',
    OUT_DIR / 'best_medial_head_by_gt_axis_field.pth',
    OUT_DIR / 'medial_training_summary.json',
    OUT_DIR / 'medial_axis_metrics_vs_epochs.png',
    OUT_DIR / 'box_sdf_medial_q_mdf_field_slices.png',
    OUT_DIR / 'pred_medial_near_sheet.ply',
    OUT_DIR / 'pred_medial_projected.ply',
    OUT_DIR / 'gt_medial_axis_box.ply',
    OUT_DIR / 'gt_medial_axis_voronoi.ply',
]
for artifact in snapshot_artifacts:
    if artifact.is_file():
        shutil.copyfile(artifact, snapshot_dir / artifact.name)

print('Saved timestamped training snapshot:', snapshot_dir)
